# short script to calculate rain amount/month 
- calculates rain amount from ICON in June and December 2023
- for comparisons in 54
- input: ICON
- output: aux/ICON_monthly_rain_amount.nc


In [1]:
import numpy as np
import xarray as xr
import os
import pandas as pd
from pathlib import Path
import sys

from tqdm.notebook import tqdm


sys.path.append("../tools")
import handle_ICON_data as ic 
import handle_OBS_data as ob # for area weights
from defs_application import bad_timesteps

import warnings

# ignore general categories
for cat in [FutureWarning, RuntimeWarning, UserWarning]:
    warnings.filterwarnings("ignore", category=cat)

# ignore specific repeated messages
for msg in [
    "pyproj unable to set PROJ database path",
    "As of v1.6.0, segmentation with time length 1",
    "Using DEFAULT_SPHERICAL_EARTH_RADIUS"
]:
    warnings.filterwarnings("ignore", message=msg)



In [2]:
def create_timestamps_withount_badones(timeperiod = None, startdate = None, enddate = None): 
    if timeperiod == "JUNE":      
        startdate = "2023-06-01-00"
        enddate =  "2023-06-30-21" #"2023-06-30-21" 
    elif timeperiod == "DECEMBER":
        startdate = "2023-12-01-00"
        enddate =  "2023-12-31-21" #"2023-06-30-21" 
    elif startdate is not None:
        startdate = startdate
        enddate =  enddate
    else: 
        print("provide JUNE, DECEMBER or startdate and enddate (YYYY-MM-DD-HH)")
        
    
    dates = ic.create_3h_timestamps(startdate, enddate)
    
    dates_array = np.array(dates)
    bad_timesteps_str = bad_timesteps.strftime('%Y-%m-%d-%H%M').tolist()
    dates = dates_array[~np.isin(dates_array, bad_timesteps_str)].tolist()
    return dates

In [11]:
# get area weights 
# set date
date = "2006-06-06-1800"

# load data
aux = ob.load_clean_crop_imerg(date,30)

def calc_area_weights(da):
    """
    Compute fractional area weights (a_grid / a_total) for a regular lat/lon grid.
    Input: xarray DataArray with lat and lon dims.
    """
    cos_lat = np.cos(np.deg2rad(da.latitude))  # (nlat,)
    weights_2d = cos_lat * xr.ones_like(da.longitude)  
    weights_2d = weights_2d / weights_2d.sum()  # a_grid / a_total
    return weights_2d

area_weights = calc_area_weights(aux)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

IMERG data loaded for 2006-06-06-1800, cleaned, cropped, 'precipitation' selected, longitude/latitude rename, assigned attrs,drop timedim


### JUNE

In [5]:
timeperiod = "JUNE"

dates = create_timestamps_withount_badones(timeperiod=timeperiod)
rain_amounts_june = []


for date in tqdm(dates, desc="Processing June dates"):
    dataset = ic.select_time(full_dataset, date)
    dataset = ic.select_vars(dataset, vlist)
    dataset = ic.add_precip_tot_rate(dataset)
    precip= ic.resample_to_lon_lat(dataset, "rain_tot_rate", zoom= zoom, to_imerg = True)

    
    # area-weighted domain mean rain rate (mm/h) (area weights sum up to 1
    weighted_mean = (precip * area_weights).sum().compute().item()
    rain_amount_june = weighted_mean * 3 # to mm
    rain_amounts_june.append(rain_amount_june)

In [6]:
# month accu
monthly_rain_amount_june = np.sum(rain_amounts_june)
print(f"{timeperiod} domain mean rain amount: {monthly_rain_amount_june} mm/month")

### DECEMEBER

In [7]:
timeperiod = "DECEMBER"
dates = create_timestamps_withount_badones(timeperiod =timeperiod)

rain_amounts_december= []

for date in tqdm(dates, desc="Processing December dates"):
    # select timestamp and variables
    dataset = ic.select_time(full_dataset, date)
    dataset = ic.select_vars(dataset, vlist)
    dataset = ic.add_precip_tot_rate(dataset)
    precip= ic.resample_to_lon_lat(dataset, "rain_tot_rate", zoom= zoom, to_imerg = True)
    
    # calc precip amount 
    weighted_mean = (precip * area_weights).sum().compute().item()
    rain_amount_december= weighted_mean * 3 # to mm
    rain_amounts_december.append(rain_amount_december)

In [8]:
# month accu
monthly_rain_amount_december = np.sum(rain_amounts_december)
print(f"{timeperiod} domain mean rain amount: {monthly_rain_amount_december} mm/month")

### SAVE

In [9]:
savedir = Path("../output/aux")
savedir.mkdir(parents=True, exist_ok=True)

monthly_rain_amount = xr.Dataset(
    {
        "rain_amount": ("timeperiod", [monthly_rain_amount_june, monthly_rain_amount_december])
    },
    coords={"timeperiod": ["JUNE", "DECEMBER"]}
)
monthly_rain_amount.attrs = {
    "description": "Monthly domain-wide accumulated rain amount for ICON, area-weighted over the tropical belt domain.",
    "source": "ICON nextGEMS ngc4008 simulation, rain_tot_rate variable",
    "units": "mm/month",
    "long_name": "domain-wide accumulated rain amount",
    "method": (
        "Area-weighted spatial mean of 3-hourly mean rain rate (rain_tot_rate), "
        "resampled to the IMERG lon-lat grid, multiplied by 3 h to obtain mm per "
        "timestep, then summed over all timestamps in the month."
    )
}


monthly_rain_amount.to_netcdf(savedir / "ICON_monthly_rain_amount.nc", engine="h5netcdf")